# Tuning de las tres mejores familias
30 configuraciones por familia y la misma CV fija. ParameterSampler cubre espacios mixtos con presupuesto fijo; tasas, C y regularización se muestrean en log, tamaños en rangos discretos.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
import pandas as pd
from src.datos import DOCS, split
from src.pipeline import make_pipeline, PhysicalFeatures


In [2]:
from src.experimentos import tuning
trials=tuning()
display(pd.DataFrame(trials).drop(columns=["fold_ap"]).sort_values("cv_ap_mean",ascending=False))

,name,phase,run_id,cv_ap_mean,cv_ap_std,train_ap,fold_train_ap_mean,cv_cost_per_1000,fit_seconds,cv_fit_seconds,gap_ap,params,warnings,learning_rate
16,hist_boosting/config_17,tuning,d520cc5a52a54790b0aa3865fc2555fc,0.878781,0.026033,0.977499,0.982198,124312.644528,0.104064,0.246067,0.098718,{'model__l2_regularization': 6.703112932069452...,0,0.103504
28,hist_boosting/config_29,tuning,d8f7f580467d48409d3cb8a160b4c7dd,0.877270,0.023921,0.999772,0.999900,123000.355451,0.144056,0.391127,0.122503,{'model__l2_regularization': 2.525452445242107...,0,0.108474
11,hist_boosting/config_12,tuning,ecb8f4a8d6484de9bc3094668f3d95cd,0.877075,0.030427,0.987890,0.996293,124563.246173,0.089953,0.241439,0.110815,{'model__l2_regularization': 0.598145791725076...,0,0.063249
19,hist_boosting/config_20,tuning,fb7dd60a87a84edab381cf331e52396f,0.876397,0.029714,0.987810,0.996546,120813.175735,0.084632,0.194224,0.111413,{'model__l2_regularization': 0.893053964287928...,0,0.103203
9,hist_boosting/config_10,tuning,2eef71647d164c17b380cc759494d402,0.876340,0.033359,0.999380,1.000000,122063.207028,0.102077,0.301792,0.123040,{'model__l2_regularization': 0.263733399338152...,0,0.06587
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,mlp/config_23,tuning,f8d468df9d6945aabb074ecdf34e06df,0.604733,0.067723,0.722680,0.651594,208063.991063,1.591546,3.427203,0.117947,"{'model__alpha': 3.661819220392426e-05, 'model...",4,constant
63,mlp/config_04,tuning,7b03636109384af9b32062f5f02d3720,0.583918,0.073202,0.698460,0.626790,214938.928770,1.896263,4.043008,0.114542,"{'model__alpha': 0.006796578090758156, 'model_...",3,constant
76,mlp/config_17,tuning,b7c925fadb374ba1a34f431a1dd602f4,0.564888,0.072064,0.684009,0.602321,217313.374088,1.928163,4.055079,0.119121,"{'model__alpha': 0.0015375920235481753, 'model...",4,constant
64,mlp/config_05,tuning,e548be811ac34474a9e1424d24dc9a99,0.538090,0.073368,0.656143,0.572014,219438.131936,1.933630,4.128924,0.118053,"{'model__alpha': 0.021368329072358756, 'model_...",3,constant


In [3]:
display(json.loads((DOCS/'plan_tuning.json').read_text(encoding='utf-8')))
from src.experimentos import spaces
print(spaces())

{'selected_families': ['hist_boosting', 'random_forest', 'mlp'],
 'configurations_per_family': 30,
 'method': 'ParameterSampler aleatorio con semilla, misma CV fija; 30 configuraciones por familia',
 'scales': 'C, alpha, var_smoothing, learning_rate y l2 en log; tamaños y profundidades discretos'}

{'logistic': {'model__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c06bce23290>, 'model__class_weight': [None, 'balanced'], 'model__max_iter': [500, 1000]}, 'random_forest': {'model__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c06e30ddf40>, 'model__max_depth': [None, 4, 6, 10, 16], 'model__min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c06b84eeb70>, 'model__max_features': ['sqrt', 0.7, 1.0], 'model__class_weight': [None, 'balanced']}, 'hist_boosting': {'model__learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c06b80e45f0>, 'model__max_iter': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c06b80e7e60>, 'model__max_leaf_nodes': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c06b80e73b0>, 'model__l2_regularization': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c06b80e78f0>, 'model__min_sa

In [4]:
defaults=json.loads((DOCS/'candidatos.json').read_text(encoding='utf-8'))
comparison=[]
for family in sorted({r['name'].split('/')[0] for r in trials}):
    initial=next(r for r in defaults if r['name']==family)
    best=max((r for r in trials if r['name'].split('/')[0]==family),key=lambda r:r['cv_ap_mean'])
    delta=best['cv_ap_mean']-initial['cv_ap_mean']
    comparison.append({'family':family,'default_ap':initial['cv_ap_mean'],'tuned_ap':best['cv_ap_mean'],'delta':delta,'fold_std':best['cv_ap_std'],'mejora_supera_std':delta>max(initial['cv_ap_std'],best['cv_ap_std'])})
display(pd.DataFrame(comparison))

,family,default_ap,tuned_ap,delta,fold_std,mejora_supera_std
0,hist_boosting,0.878331,0.878781,0.000450,0.026033,False
1,mlp,0.753560,0.791341,0.037781,0.060867,False
2,random_forest,0.868115,0.873025,0.004910,0.033908,False


Una diferencia menor que la desviación entre folds no se interpreta como mejora demostrada. La repetición de selección y tuning en los mismos folds puede producir optimismo; no equivale a CV anidada. Las corridas hijas registran parámetros, métricas, warnings y pipeline completo con firma.